<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">  <tr style="border: none;">    <td style="vertical-align: middle; border: none; padding: 15px 20px;">      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">        ✂️ 02. Árboles de Regresión y Poda (Pruning)      </h1>      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">        Especialización en Ciencia de Datos | Programación para Ciencia de Datos      </p>      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">        Universidad Santo Tomás — Seccional Tunja      </p>    </td>    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">        💡 Para Dummies • Módulo 09      </span><br>      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>    </td>  </tr></table><div align="center" style="margin-top: 15px; margin-bottom: 15px;">  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/02_Arboles_Regresion_y_Poda_Cost_Complexity_Dummies.ipynb" target="_parent">    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>  </a></div>

---
## ¿Qué vamos a aprender aquí? 🎈

Hasta ahora nuestros árboles predecían **categorías** ("Aprueba"/"Reprueba", "Spam"/"Legítimo"). Pero un árbol también puede predecir **números** (por ejemplo, el precio de una casa). Y cuando dejamos que un árbol crezca sin límite, aparece un problema clásico: **memoriza** los datos de entrenamiento en vez de **aprender un patrón general**.

En este cuaderno verás, con un ejemplo pequeño y controlado:
1. Cómo un árbol de regresión predice números (no categorías).
2. Qué es el sobreajuste (*overfitting*) y por qué es peligroso.
3. Cómo la **poda** (*pruning*) soluciona ese problema, con el parámetro `ccp_alpha`.

---
## 1. Predecir un número: el promedio del vecindario 🏘️

Cuando el árbol clasifica, cada hoja "vota" por la categoría más común del grupo. Cuando el árbol hace **regresión** (predice un número), cada hoja simplemente entrega el **promedio** de los valores numéricos que cayeron ahí.

Es como si un tasador de casas dijera: *"las casas de este vecindario, con esta cantidad de metros cuadrados, en promedio cuestan $250 millones"* — y usara ese promedio como su predicción para cualquier casa parecida, sin importar que ninguna casa cueste exactamente eso.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

np.random.seed(3)

print("Entorno listo para explorar árboles de regresión y poda.")

---
## 2. Creamos un mini-set de precios de casas 🏠

Inventamos 30 casas ficticias con una sola característica de entrada, `area_m2` (área en metros cuadrados), y el precio (`precio_millones`) que queremos predecir. Usamos una relación creciente con algo de ruido, para que se parezca a un mercado real (casas más grandes cuestan más, pero no de forma perfectamente exacta).

In [ ]:
n = 30
area_m2 = np.round(np.random.uniform(40, 220, n), 1)
precio_millones = 80 + 1.1 * area_m2 + np.random.normal(0, 25, n)

casas = pd.DataFrame({'area_m2': area_m2, 'precio_millones': np.round(precio_millones, 1)})
casas = casas.sort_values('area_m2').reset_index(drop=True)

casas.head(8)

### 🤔 ¿Qué acaba de pasar?

Creamos 30 casas ficticias donde el precio depende principalmente del área, pero con ruido aleatorio (`np.random.normal`) para simular otros factores que no estamos midiendo (ubicación, antigüedad, acabados...). Ordenamos por área solo para que la tabla se lea más fácil — el árbol no necesita que los datos estén ordenados.

---
## 3. Un árbol de regresión sin restricciones: el peligro de memorizar 📈

Vamos a entrenar un `DecisionTreeRegressor` **sin poner ningún límite** de profundidad, y a separar nuestras casas en un grupo de entrenamiento y uno de prueba, para ver qué tan bien generaliza el árbol a casas que nunca vio.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

X = casas[['area_m2']]
y = casas['precio_millones']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

arbol_libre = DecisionTreeRegressor(random_state=42)
arbol_libre.fit(X_train, y_train)

print(f"R² en entrenamiento (sin restricción): {arbol_libre.score(X_train, y_train):.3f}")
print(f"R² en prueba (sin restricción):        {arbol_libre.score(X_test, y_test):.3f}")
print(f"Número de hojas del árbol: {arbol_libre.get_n_leaves()}")

### 🤔 ¿Qué acaba de pasar?

- `R²` mide qué tan bien predice el modelo (1.0 es perfecto, valores más bajos o negativos son peores).
- El `R²` en **entrenamiento** suele salir cercano a 1.0 — casi perfecto — porque el árbol, al no tener límite de profundidad, sigue preguntando hasta que cada hoja tiene una sola casa (o muy pocas). Literalmente **memorizó** el precio exacto de cada casa de entrenamiento.
- Pero el `R²` en **prueba** (casas que el árbol nunca vio) es notablemente peor. El árbol no aprendió el patrón general "a más área, más precio" — aprendió los datos de memoria, ruido incluido. A esto se le llama **sobreajuste (*overfitting*)**: es como un estudiante que memoriza las respuestas exactas de un examen de práctica, pero no entiende el tema, y por eso le va mal en el examen real con preguntas distintas.

---
## 4. Podar el árbol: la analogía del jardinero 🌳✂️

Un jardinero que deja crecer un árbol sin ningún control termina con ramas por todas partes, muchas de ellas inútiles. Para que el árbol crezca sano y compacto, el jardinero lo **poda**: corta las ramas que no aportan mucho.

Scikit-Learn hace lo mismo con el parámetro **`ccp_alpha`** (poda por complejidad de costo, *Cost-Complexity Pruning*): entre más alto el valor de `ccp_alpha`, más "tijera" le metemos al árbol — se eliminan las ramas que reducen muy poco el error, a cambio de tener un árbol más simple y que generaliza mejor.

In [ ]:
# Calculamos algunos valores candidatos de alpha para podar
ruta = arbol_libre.cost_complexity_pruning_path(X_train, y_train)
alphas_candidatos = ruta.ccp_alphas

# Probamos un puñado de valores de alpha y medimos el R² en prueba para cada uno
alphas_muestra = np.linspace(alphas_candidatos.min(), alphas_candidatos[int(len(alphas_candidatos) * 0.8)], 10)
r2_prueba = []

for a in alphas_muestra:
    modelo = DecisionTreeRegressor(ccp_alpha=a, random_state=42)
    modelo.fit(X_train, y_train)
    r2_prueba.append(modelo.score(X_test, y_test))

mejor_alpha = alphas_muestra[np.argmax(r2_prueba)]
print(f"Mejor alpha encontrado: {mejor_alpha:.3f}")
print(f"R² en prueba con ese alpha: {max(r2_prueba):.3f}  (vs. {arbol_libre.score(X_test, y_test):.3f} sin podar)")

### 🤔 ¿Qué acaba de pasar?

- `cost_complexity_pruning_path` le pregunta al propio árbol: *"¿qué valores de `ccp_alpha` tendrían algún efecto sobre tus ramas?"* y nos devuelve una lista de candidatos ordenada de menor a mayor poda.
- Probamos varios de esos valores, entrenando un árbol nuevo con cada uno, y nos quedamos con el que mejor generaliza en el conjunto de prueba (mayor `R²`).
- Compara el `R²` en prueba del árbol podado contra el del árbol libre del paso anterior: normalmente el árbol podado generaliza igual o mejor, aunque en entrenamiento ya no obtenga un `R²` perfecto — y eso es justo lo que queremos: un modelo que funcione bien con casas nuevas, no uno que memorice las que ya vio.

In [ ]:
arbol_podado = DecisionTreeRegressor(ccp_alpha=mejor_alpha, random_state=42)
arbol_podado.fit(X_train, y_train)

plt.figure(figsize=(10, 5), dpi=110)
plot_tree(
    arbol_podado,
    feature_names=['area_m2'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title(f"Árbol de precios ya podado ({arbol_podado.get_n_leaves()} hojas)", fontweight='bold')
plt.show()

### 🤔 ¿Qué acaba de pasar?

Compara este árbol con el que tendrías sin podar (que fácilmente tendría tantas hojas como casas de entrenamiento). El árbol podado tiene muchas menos hojas, y cada hoja ahora agrupa **varias** casas parecidas en área, prediciendo el `value` (el precio promedio de ese grupo) en lugar del precio exacto de una sola casa. Menos hojas = reglas más simples = mejor generalización, siempre y cuando no podemos "de más" (si podamos demasiado, el árbol se vuelve tan simple que deja de capturar el patrón real).

---
## 5. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| `DecisionTreeRegressor` | Versión del árbol para predecir números; cada hoja entrega el promedio de sus casos. |
| Sobreajuste (*overfitting*) | El árbol memoriza los datos de entrenamiento (ruido incluido) y generaliza mal a datos nuevos. |
| `ccp_alpha` | Parámetro de poda: entre más alto, más ramas se eliminan y más simple queda el árbol. |
| `cost_complexity_pruning_path` | Método que sugiere valores candidatos de `ccp_alpha` a partir del propio árbol entrenado. |
| Poda (*pruning*) | Recortar ramas poco útiles para lograr un árbol más simple que generaliza mejor — como podar un jardín. |

➡️ **Siguiente paso:** el próximo cuaderno de esta serie Para Dummies — **"03 - Métodos de Ensamble: Bagging y Random Forests (Para Dummies)"** — está siendo preparado y explicará, con la misma calma, cómo combinar **muchos** árboles pequeños para armar modelos todavía más precisos y estables.

---<div align="center">  <p style="font-size: 0.9em; color: #64748b;">    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>  </p></div>